This notebook is a simple example of bounding an experimental design and simulating a few replicates.

# Imports

In [1]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import dask.dataframe as dd
 

%load_ext autoreload
%autoreload 2

2025-09-22 10:47:32.564461: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-22 10:47:32.568689: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

# Create cluster

In [2]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='24GB')
client=Client(cluster)

2025-09-22 10:47:40,483 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 24GB due to system memory limit of 16.00 GiB
2025-09-22 10:47:40,485 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 24GB due to system memory limit of 16.00 GiB
2025-09-22 10:47:40,487 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 24GB due to system memory limit of 16.00 GiB
2025-09-22 10:47:40,489 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 24GB due to system memory limit of 16.00 GiB


# Seting experimiental design parameters

In [9]:
#first, we define the new parameters we want to assign to this object.

new_cell_number=pd.Series({"reference":500,"blood":500})

#we make up 3x replicates
new_zi=pd.Series({"replicate_A":0.02,"replicate_B":0.021})

new_min=1
new_max=200

new_MOI=60

In [10]:
#next, let's create a bounds object from these parameters and the bounds of the shendure data.
artificial_bounds=scm.SHENDURE_BOUNDS.copy(
    min_mpra_umi=new_min,
    max_mpra_umi=new_max,
    zi=new_zi,
    cells_per_cell_type=new_cell_number)
    
artificial_bounds.set_effective_moi(new_MOI)

# Creating an artificial library

In [11]:
#making up the CREs
spread_gt,spread_hypothesis=scm.simple_spread(cell_types=new_cell_number.keys(),
                  min=new_min,
                  max=new_max,
                  fineness=2)

library=scm.simulate_library(CREs=spread_gt["cre_id"],
                 library_model=artificial_bounds.library_model)

In [12]:
library

,cre_id,mpra_bc,abundance
0,reference,AAAAAAAAAAAAAAAAAAAA,0.000345
1,reference,AAAAAAAAAAAAAAAAAAAC,0.001315
2,reference,AAAAAAAAAAAAAAAAAAAG,0.000445
3,reference,AAAAAAAAAAAAAAAAAAAT,0.000309
4,reference,AAAAAAAAAAAAAAAAAACA,0.001735
...,...,...,...
1013,CRE_even_1_high_in_blood,AAAAAAAAAAAAAAATTTCC,0.001273
1014,CRE_even_1_high_in_blood,AAAAAAAAAAAAAAATTTCG,0.000176
1015,CRE_even_1_high_in_blood,AAAAAAAAAAAAAAATTTCT,0.001823
1016,CRE_even_1_high_in_blood,AAAAAAAAAAAAAAATTTGA,0.000689


# Performing de-novo simulation

In [13]:
batch_plus=scm.de_novo_simulation(
                        simulation_replicates=3,
                        experiment_bounds=artificial_bounds,
                        ground_truth=spread_gt,
                        library=library)

In [14]:
batch_plus.gamut(client)

# Save

In [15]:
data_root="/gpfs/gibbs/pi/reilly/tabula_data"

In [16]:
batch_plus.save(data_root,"simulated/tiny_de_novo")

In [17]:
spread_hypothesis.to_tsv(f"{data_root}/simulated/tiny_de_novo_hypotheses.tsv")

In [18]:
cluster.close()

2025-09-22 10:52:44,181 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:43031' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'_wrap_helper-2566bab3391e476f27bd82fca63bf372'} (stimulus_id='handle-worker-cleanup-1758552764.181423')
2025-09-22 10:52:44,184 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:35851' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'_wrap_helper-61b0473fe7eb8520fd648c534cae5d48'} (stimulus_id='handle-worker-cleanup-1758552764.1841128')
2025-09-22 10:52:44,185 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:34921' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'_wrap_helper-81ca91ba9c37e34ab7381b7cac86aa0c'} (stimulus_id='handle-worker-cleanup-1758552764.185482')
